In [70]:
import yaml
import os
import polars as pl
from datetime import datetime 

with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

data_dir = config["data_dir"]
print(data_dir  )
if os.path.exists(data_dir) and os.path.isdir(data_dir):
    list_of_files = os.listdir(data_dir)
else:
    list_of_files = []
print(list_of_files)

E:\smallcaps_data
['AIXI.parquet', 'APUS.parquet', 'BAOS.parquet', 'BEAT.parquet', 'BIAF.parquet', 'CAN.parquet', 'CDTG.parquet', 'CHAI.parquet', 'CHPT.parquet', 'CISS.parquet', 'COOT.parquet', 'CYAB.parquet', 'FBGL.parquet', 'FLYE.parquet', 'GCDT.parquet', 'GIPR.parquet', 'GRI.parquet', 'GRML.parquet', 'IMCC.parquet', 'IMRN.parquet', 'LIDR.parquet', 'logs', 'MEDS.parquet', 'NCPL.parquet', 'NCRA.parquet', 'NEOV.parquet', 'OFAL.parquet', 'OLB.parquet', 'OLOX.parquet', 'PDSB.parquet', 'PFSA.parquet', 'SBFM.parquet', 'SDEV.parquet', 'SKYQ.parquet', 'SOAR.parquet', 'SPHL.parquet', 'STAK.parquet', 'TNMG.parquet', 'TOPP.parquet', 'TRUG.parquet', 'USDE.parquet', 'VEEA.parquet', 'VRA.parquet', 'VRME.parquet', 'XHLD.parquet', 'XPON.parquet', 'XTNT.parquet', '_catalog.parquet']


In [71]:
list_of_files = [f for f in list_of_files if f.endswith(".parquet")]
list_of_files=[f for f in list_of_files if f.startswith("VEEA")]
list_of_files

['VEEA.parquet']

In [72]:
df = [pl.read_parquet(os.path.join(data_dir, file)) for file in list_of_files if file.endswith(".parquet")]


In [73]:
df[0].filter(pl.col('datetime').dt.date()==datetime(2026,10,5).date())


datetime,open,high,low,close,volume,spread,transactions
"datetime[μs, UTC]",f64,f64,f64,f64,i64,f64,i64
2026-10-05 08:00:00 UTC,4.31,4.7406,4.15,4.15,78893,null,1076
2026-10-05 08:01:00 UTC,4.15,4.58,4.15,4.1506,35747,null,421
2026-10-05 08:02:00 UTC,4.1593,4.76,4.1304,4.39,381615,null,6896
2026-10-05 08:03:00 UTC,4.39,4.49,4.1,4.1004,63551,null,903
2026-10-05 08:04:00 UTC,4.1005,4.11,4.091,4.091,13072,null,168
…,…,…,…,…,…,…,…
2026-10-05 23:55:00 UTC,5.39,5.41,5.39,5.39,12498,null,20
2026-10-05 23:56:00 UTC,5.39,5.39,5.3,5.3077,6648,null,35
2026-10-05 23:57:00 UTC,5.35,5.39,5.3,5.39,3816,null,27


In [ ]:
df_to_plot = (
    df[0]
    .with_columns(
        pl.col("datetime").dt.convert_time_zone("America/New_York").alias("datetime")
    )
    .sort("datetime")
    .with_columns(
        pl.col("datetime").dt.date().alias("trading_day"),

        ((pl.col("high") + pl.col("low") + pl.col("close")) / 3)
        .alias("price"),
    )
    .with_columns(
        (
            (pl.col("price") * pl.col("volume")).cum_sum().over("trading_day")
            / pl.col("volume").cum_sum().over("trading_day")
        ).alias("vwap_daily")
    )
).filter(pl.col('datetime').dt.date()==datetime(2026,10,5).date())
print(df_to_plot)

shape: (950, 11)
┌──────────────┬────────┬────────┬────────┬───┬──────────────┬─────────────┬──────────┬────────────┐
│ datetime     ┆ open   ┆ high   ┆ low    ┆ … ┆ transactions ┆ trading_day ┆ price    ┆ vwap_daily │
│ ---          ┆ ---    ┆ ---    ┆ ---    ┆   ┆ ---          ┆ ---         ┆ ---      ┆ ---        │
│ datetime[μs, ┆ f64    ┆ f64    ┆ f64    ┆   ┆ i64          ┆ date        ┆ f64      ┆ f64        │
│ America/New_ ┆        ┆        ┆        ┆   ┆              ┆             ┆          ┆            │
│ York]        ┆        ┆        ┆        ┆   ┆              ┆             ┆          ┆            │
╞══════════════╪════════╪════════╪════════╪═══╪══════════════╪═════════════╪══════════╪════════════╡
│ 2026-10-05   ┆ 4.31   ┆ 4.7406 ┆ 4.15   ┆ … ┆ 1076         ┆ 2026-10-05  ┆ 4.346867 ┆ 4.346867   │
│ 04:00:00 EDT ┆        ┆        ┆        ┆   ┆              ┆             ┆          ┆            │
│ 2026-10-05   ┆ 4.15   ┆ 4.58   ┆ 4.15   ┆ … ┆ 421          ┆ 2026-10-05 

In [85]:
from datetime import datetime, time
from zoneinfo import ZoneInfo

import plotly.graph_objects as go
from plotly.subplots import make_subplots

NEW_YORK = ZoneInfo("America/New_York")

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.75, 0.25]
)

fig.add_trace(
    go.Candlestick(
        x=df_to_plot["datetime"],
        open=df_to_plot["open"],
        high=df_to_plot["high"],
        low=df_to_plot["low"],
        close=df_to_plot["close"],
        name="Precio"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=df_to_plot["datetime"],
        y=df_to_plot["vwap_daily"],
        mode="lines",
        name="VWAP"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(
        x=df_to_plot["datetime"], y=df_to_plot["volume"], name="Volumen"
    ),
    row=2,
    col=1
)

market_dates = (
    df_to_plot.select(pl.col("datetime").dt.date().unique().sort())
    .to_series()
    .to_list()
)
for market_date in market_dates:
    regular_session_start = datetime.combine(
        market_date, time(9, 30), tzinfo=NEW_YORK
    )
    regular_session_end = datetime.combine(
        market_date, time(16, 0), tzinfo=NEW_YORK
    )
    fig.add_vrect(
        x0=regular_session_start,
        x1=regular_session_end,
        fillcolor="lightgreen",
        opacity=0.12,
        line_width=0,
        layer="below",
        row="all",
        col=1
    )

fig.update_layout(
    title="Precio + VWAP + Volumen",
    xaxis_rangeslider_visible=False,
    height=800
)

fig.show()

In [77]:
list_of_files


['VEEA.parquet']

In [80]:
df_catalogo = pl.read_parquet(os.path.join(data_dir, "_catalog.parquet"))
df_catalogo

ticker,start,end,bars,updated_at
str,"datetime[μs, UTC]","datetime[μs, UTC]",i64,"datetime[μs, UTC]"
"""AIXI""",2026-07-10 08:00:00 UTC,2026-08-07 23:59:00 UTC,5346,2026-09-20 17:41:06.726497 UTC
"""AIXI""",2026-08-10 08:00:00 UTC,2026-08-28 23:59:00 UTC,7201,2026-09-20 17:41:07.784659 UTC
"""APUS""",2026-08-06 13:30:00 UTC,2026-09-04 20:32:00 UTC,1205,2026-09-26 16:49:35.748018 UTC
"""APUS""",2026-09-08 13:30:00 UTC,2026-09-24 23:59:00 UTC,1817,2026-09-26 16:49:56.532342 UTC
"""APUS""",2026-09-18 08:56:00 UTC,2026-10-07 23:59:00 UTC,4454,2026-10-09 18:06:54.554175 UTC
…,…,…,…,…
"""XHLD""",2026-09-17 08:25:00 UTC,2026-10-06 23:59:00 UTC,7460,2026-10-09 18:08:18.905421 UTC
"""XPON""",2026-07-06 08:00:00 UTC,2026-08-04 20:05:00 UTC,1417,2026-09-20 17:38:04.108728 UTC
"""XPON""",2026-08-05 13:30:00 UTC,2026-08-24 23:59:00 UTC,2487,2026-09-20 17:38:05.088360 UTC


In [84]:
df_catalogo.filter(pl.col('ticker')=="VEEA")['end'].dt.date()

end
date
2026-08-25
2026-09-18
2026-09-15
2026-10-08
2026-10-09
